# Arplio Voice — engine test

Hear which voice engines are good enough to sell. You upload one short clip of
a voice, and this notebook clones it with every engine side by side.

**Why:** the default engine (OmniVoice) is free for personal use only — its
licence forbids commercial use. The engines below *do* allow commercial use;
this test shows whether their quality is good enough.

**Before you start:** `Runtime → Change runtime type → T4 GPU` (free).
Then `Runtime → Run all`. Total time: about 30–45 minutes, mostly downloads.
When cell 4 asks, upload your clip.

| Engine | Licence | Sell with it? | Languages |
|---|---|---|---|
| OmniVoice (baseline) | CC-BY-NC | ❌ personal only | 646, incl. Yoruba, Hausa, Igbo, Pidgin |
| VoxCPM2 | Apache-2.0 | ✅ | 30 (incl. Swahili) |
| CosyVoice 3 | Apache-2.0 | ✅ | ~9 major languages |
| MOSS-TTS-Nano | Apache-2.0 | ✅ | 20 |
| PocketTTS | CC-BY-4.0 | ✅ with credit | 6 European (English here) |

## 1–3 · Set up (runs by itself)

In [ ]:
# ── 1. GPU check ────────────────────────────────────────────────────────────
# Confirms the runtime has an NVIDIA GPU. Everything still works on CPU, but
# a single sentence can take minutes instead of seconds.
import shutil
import subprocess

if shutil.which("nvidia-smi"):
    print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
    print("GPU detected — you're good to go.")
else:
    print("=" * 74)
    print("WARNING: no NVIDIA GPU in this runtime.")
    print("Fix: menu bar -> Runtime -> Change runtime type -> T4 GPU,")
    print("then re-run this notebook from the top.")
    print("(Continuing anyway works, but generation will be very slow on CPU.)")
    print("=" * 74)

In [ ]:
# ── 2. Clone + install (idempotent; first run ~5-8 min) ─────────────────────
import os
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/Goldenjohnny-art/arpliovoice"
REPO_DIR = "/content/VoiceStudio"
BUN = "/root/.bun/bin/bun"

# Generous network timeouts: Colab -> PyPI is usually fast, but model/wheel
# CDNs occasionally stall and uv's default timeout is short.
os.environ.setdefault("UV_HTTP_TIMEOUT", "300")

def run(cmd, *, cwd=None, what=""):
    """Stream a command's output; stop the cell with an actionable message on failure."""
    shown = cmd if isinstance(cmd, str) else " ".join(cmd)
    print(f"\n$ {shown}")
    rc = subprocess.run(cmd, cwd=cwd, shell=isinstance(cmd, str)).returncode
    if rc != 0:
        raise SystemExit(
            f"\nFAILED: {what or shown} (exit code {rc}).\n"
            "Scroll up for the underlying error. Re-running this cell is safe.\n"
            "Still stuck? Open an issue with the output above:\n"
            "  https://github.com/Goldenjohnny-art/arpliovoice/issues"
        )

# 2a. Clone the repo (reused if already present)
if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    print(f"Repo already present at {REPO_DIR} — reusing it.")
else:
    run(["git", "clone", "--depth", "1", "--branch", "arplio-voice", REPO_URL, REPO_DIR], what="git clone")

# 2b. System packages (ffmpeg: audio/video processing; libsndfile1: soundfile)
run("apt-get -qq update && apt-get -qq install -y ffmpeg libsndfile1", what="apt-get install")

# 2c. bun — builds the web UI (see the intro cell for why we build it here)
if not os.path.exists(BUN):
    run("curl -fsSL https://bun.sh/install | bash", what="bun installer")
os.environ["PATH"] = os.path.dirname(BUN) + os.pathsep + os.environ["PATH"]
run([BUN, "--version"], what="bun version check")

# 2d. Build the frontend (skipped once frontend/dist/index.html exists)
dist_index = os.path.join(REPO_DIR, "frontend", "dist", "index.html")
if os.path.exists(dist_index):
    print("frontend/dist already built — skipping.")
else:
    # The repo is a bun workspace: install at the repo root, build the Electron web renderer.
    run([BUN, "install", "--frozen-lockfile"], cwd=REPO_DIR, what="bun install (workspace)")
    run([BUN, "run", "--cwd", "electron", "build:web"], cwd=REPO_DIR, what="Electron web renderer build (vite)")
    if not os.path.exists(dist_index):
        raise SystemExit(
            "The frontend build finished but frontend/dist/index.html is missing —\n"
            "scroll up for vite errors, then re-run this cell."
        )

# 2e. Backend dependencies — same command the official Docker image uses.
if not shutil.which("uv"):
    run([sys.executable, "-m", "pip", "install", "-q", "uv"], what="pip install uv")
# --constraint: `uv pip install` ignores pyproject's `[tool.uv]
# constraint-dependencies` (a project-API setting), so without it torch,
# torchaudio and torchvision resolve on their bare lower bounds and Colab's
# preinstalled torchvision can be left behind a newer torch -- which fails at
# import with "operator torchvision::nms does not exist" (#1357).
run(["uv", "pip", "install", "--system", "--no-cache",
     "--constraint", "deploy/torch-constraints.txt", "."],
    cwd=REPO_DIR, what="backend install (uv pip install --system .)")

# 2f. cuDNN 8 side-install for CTranslate2 (WhisperX ASR on GPU). PyTorch 2.8+
# ships cuDNN 9; scripts/setup.py places cuDNN 8 libs where the backend
# preloads them from (<repo>/.venv/.../cudnn8_compat). We create the .venv
# directory only so the script has its expected target — no actual venv is
# used on Colab. Non-fatal: without it, transcription falls back to the
# torch-native Whisper backend automatically.
os.makedirs(os.path.join(REPO_DIR, ".venv"), exist_ok=True)
run([sys.executable, os.path.join(REPO_DIR, "scripts", "setup.py")], what="cuDNN 8 compat setup")

# 2g. Sanity check in a fresh interpreter (this kernel may hold a stale torch).
# Imports the backend's own model stack, not just torch — Colab's system
# Python mixes preinstalled and freshly-resolved wheels, and a torchaudio or
# transformers that can't load together only shows up when the model module is
# imported (#1229). Catching it here beats a 5-minute health timeout in cell 5.
run([sys.executable, "-c",
     # Versions FIRST: if the model-stack import below fails, the cell output
     # still shows what was actually installed, which is the single most
     # useful line for diagnosing a Colab environment (#1229).
     "import torch, torchaudio, uvicorn, fastapi, transformers; "
     "print(f'torch {torch.__version__}, torchaudio {torchaudio.__version__}, "
     "transformers {transformers.__version__}, "
     "CUDA available: {torch.cuda.is_available()}'); "
     "from omnivoice.models.omnivoice import OmniVoice; "
     "from transformers import HiggsAudioV2TokenizerModel; "
     "print('Install OK - backend model stack imports cleanly')"],
    cwd=REPO_DIR, what="import sanity check")

In [ ]:
# ── 5. Launch the backend ───────────────────────────────────────────────────
# Starts uvicorn on port 3900 and waits for /health. Re-running this cell when
# the backend is already up just reports its status.
import json
import os
import subprocess
import sys
import time
import urllib.request

REPO_DIR = "/content/VoiceStudio"
PORT = 3900
LOG_PATH = "/content/omnivoice_backend.log"
HEALTH_URL = f"http://127.0.0.1:{PORT}/health"

def health():
    try:
        with urllib.request.urlopen(HEALTH_URL, timeout=5) as r:
            return json.load(r)
    except Exception:
        return None

info = health()
if info:
    print(f"Backend already running — {info}")
else:
    # Clear any half-dead process from a previous run of this cell.
    subprocess.run(f"kill -9 $(lsof -t -i:{PORT}) 2>/dev/null || true", shell=True)
    time.sleep(1)

    env = os.environ.copy()
    # Headless-server deployment, same as the official Docker image: relaxes
    # the desktop-only loopback origin gate so the proxied browser session can
    # reach the settings/system routes.
    env["OMNIVOICE_SERVER_MODE"] = "1"
    # Voices, projects, and generated audio live here. Ephemeral! See the
    # troubleshooting cell for persisting it to Google Drive.
    env["OMNIVOICE_DATA_DIR"] = "/content/omnivoice_data"
    env["PYTHONUNBUFFERED"] = "1"

    log = open(LOG_PATH, "ab")
    proc = subprocess.Popen(
        [sys.executable, "-m", "uvicorn", "main:app",
         "--app-dir", "backend", "--host", "127.0.0.1", "--port", str(PORT)],
        cwd=REPO_DIR, env=env, stdout=log, stderr=subprocess.STDOUT,
    )
    print(f"Backend starting (PID {proc.pid}); log: {LOG_PATH}")

    deadline = time.time() + 300
    while time.time() < deadline:
        if proc.poll() is not None:
            break  # process died — report below
        info = health()
        if info:
            break
        print(".", end="", flush=True)
        time.sleep(3)
    print()

    if info:
        print(f"Backend is up — {info}")
        if "cuda" not in str(info.get("device", "")):
            print("NOTE: device is not CUDA — generation will be slow. "
                  "See cell 1 to enable the GPU runtime.")
    else:
        try:
            with open(LOG_PATH, "r", errors="replace") as f:
                tail = "".join(f.readlines()[-40:])
        except OSError:
            tail = "(no log file found)"
        raise SystemExit(
            "Backend did not become healthy within 5 minutes.\n"
            f"--- last lines of {LOG_PATH} ---\n{tail}\n"
            "--- end of log ---\n"
            "Fix the error above (usually a missing dependency: re-run cell 2),\n"
            "then re-run this cell. Still stuck? Attach the log to an issue:\n"
            "  https://github.com/Goldenjohnny-art/arpliovoice/issues"
        )

## 4 · Your voice

Upload a **clean 5–15 second clip of one person talking** (WAV, MP3 or M4A — no
music, little background noise), and type exactly what is said in it.

In [ ]:
# ── 4. Upload the voice to clone ────────────────────────────────────────────
import os
from google.colab import files

REF_TEXT = ""  #@param {type:"string"}
#@markdown ↑ Exactly what is said in your clip (leave empty to auto-transcribe).
TEST_TEXT = "Welcome to Arplio Voice. Turn any script into a natural voiceover, in seconds, for your next video."  #@param {type:"string"}

uploaded = files.upload()
REF_PATH = os.path.abspath(next(iter(uploaded)))
print("Reference clip:", REF_PATH)

## 5 · Install the engines

Each engine installs into its own folder with its own model download
(VoxCPM2 ≈ 5 GB, CosyVoice 3 ≈ 10 GB). Untick any you want to skip.

In [ ]:
# ── 5. Install the commercially licensed engines ────────────────────────────
import time
import requests

BASE = "http://127.0.0.1:3900"
VOXCPM2 = True  #@param {type:"boolean"}
COSYVOICE3 = True  #@param {type:"boolean"}
MOSS_TTS_NANO = True  #@param {type:"boolean"}
POCKETTTS = True  #@param {type:"boolean"}

WANTED = [e for e, on in (("voxcpm2", VOXCPM2), ("cosyvoice", COSYVOICE3),
                          ("moss-tts-nano", MOSS_TTS_NANO), ("pockettts", POCKETTTS)) if on]

def status(engine):
    return requests.get(f"{BASE}/engines/sidecar/{engine}/install/status", timeout=30).json()

for engine in WANTED:
    r = requests.post(f"{BASE}/engines/sidecar/{engine}/install", timeout=60)
    print(f"{engine}: {r.json() if r.ok else r.text[:300]}")

RESULTS = {}
pending = set(WANTED)
while pending:
    for engine in sorted(pending):
        s = status(engine)
        job = s.get("job") or {}
        if s.get("installed") and job.get("state") not in ("running", "queued"):
            RESULTS[engine] = "installed"
        elif job.get("state") in ("failed", "error", "cancelled"):
            RESULTS[engine] = "FAILED: " + (job.get("error") or "")[-400:]
        else:
            step = next((st["id"] for st in job.get("steps", []) if st.get("state") == "running"), "…")
            print(f"  {engine}: {step}", end="   ")
            continue
        pending.discard(engine)
    print()
    if pending:
        time.sleep(20)

for engine, result in RESULTS.items():
    print(f"{engine:14s} {result}")

## 6 · Listen and compare

The same sentence, in the same cloned voice, from every engine. Listen for:
**does it sound like the person**, is it natural, and are there glitches.

In [ ]:
# ── 6. Clone the voice with every engine ────────────────────────────────────
import os
import time
import requests
from IPython.display import Audio, Markdown, display

BASE = "http://127.0.0.1:3900"
OUT = "/content/arplio_engine_test"
os.makedirs(OUT, exist_ok=True)

LICENCE = {
    "omnivoice": "CC-BY-NC — personal use only (baseline)",
    "voxcpm2": "Apache-2.0 — commercial OK",
    "cosyvoice": "Apache-2.0 — commercial OK",
    "moss-tts-nano": "Apache-2.0 — commercial OK",
    "pockettts": "CC-BY-4.0 — commercial OK with credit",
}

display(Markdown("**Your original clip**"))
display(Audio(REF_PATH))

for engine in ["omnivoice", *WANTED]:
    form = {"text": TEST_TEXT, "language": "English", "engine": engine}
    if REF_TEXT.strip():
        form["ref_text"] = REF_TEXT.strip()
    started = time.time()
    with open(REF_PATH, "rb") as ref:
        r = requests.post(f"{BASE}/generate", data=form,
                          files={"ref_audio": (os.path.basename(REF_PATH), ref)}, timeout=1800)
    seconds = time.time() - started
    display(Markdown(f"### {engine} · {LICENCE[engine]}"))
    if r.status_code != 200:
        print(f"Could not generate ({r.status_code}): {r.text[:600]}")
        continue
    path = os.path.join(OUT, f"{engine}.wav")
    with open(path, "wb") as f:
        f.write(r.content)
    print(f"Took {seconds:.1f}s on this GPU")
    display(Audio(path))

## 7 · (Optional) Nigerian languages — OmniVoice only

None of the commercially licensed engines speak Yoruba, Hausa, Igbo or Pidgin.
This shows what OmniVoice does with them, so you know what you would gain by
asking its authors for a commercial licence.

In [ ]:
# ── 7. Nigerian languages with OmniVoice (non-commercial licence) ───────────
SAMPLES = {
    "Yoruba": "Ẹ kú àbọ̀ sí Arplio. A ó ràn yín lọ́wọ́ láti ṣe fídíò yín.",
    "Nigerian Pidgin": "Welcome to Arplio. We go help you make your video sweet well well.",
    "Hausa": "Barka da zuwa Arplio. Za mu taimaka maka ka yi bidiyonka.",
}
for language, sentence in SAMPLES.items():
    with open(REF_PATH, "rb") as ref:
        r = requests.post(f"{BASE}/generate",
                          data={"text": sentence, "language": language, "engine": "omnivoice"},
                          files={"ref_audio": (os.path.basename(REF_PATH), ref)}, timeout=1800)
    display(Markdown(f"### {language}"))
    if r.status_code != 200:
        print(f"Could not generate ({r.status_code}): {r.text[:400]}")
        continue
    path = os.path.join(OUT, f"omnivoice_{language.replace(' ', '_').lower()}.wav")
    with open(path, "wb") as f:
        f.write(r.content)
    display(Audio(path))

In [ ]:
# ── 8. Download every clip as one zip ───────────────────────────────────────
import shutil
from google.colab import files

archive = shutil.make_archive("/content/arplio_engine_test", "zip", "/content/arplio_engine_test")
files.download(archive)